##### District

In [1]:
import os
import pandas as pd
import numpy as np
import random

def convert_files_with_depot(input_folder, output_folder, date_str="20240301"):
    os.makedirs(output_folder, exist_ok=True)

    for i, filename in enumerate(sorted(os.listdir(input_folder))):
        if filename.endswith(".csv"):
            input_path = os.path.join(input_folder, filename)
            df = pd.read_csv(input_path)

            # 計算車輛數量
            num_shops = df.shape[0]
            nv = max(1, num_shops // random.randint(10, 15))

            # 隨機 Demand（0-15）
            df["Demand"] = np.random.randint(0, 16, size=len(df))

            # 其他欄位補齊
            df["Date"] = date_str
            df["Number"] = df["商店編號"]
            df["Name"] = df["name"]
            df["W_S"] = 0
            df["W_F"] = 86400
            df["Service_time"] = 600
            df["N_V"] = nv
            df["C_V1"] = 1800
            df["C_V2"] = 1800
            df["C_V3"] = 1800

            df = df[["Date", "Number", "Name", "X", "Y", "Demand", "W_S", "W_F", "Service_time", "N_V", "C_V1", "C_V2", "C_V3"]]

            # ✅ 插入瑞芳 depot 為第 0 筆資料
            depot = {
                "Date": date_str,
                "Number": 0,
                "Name": "瑞芳",
                "X": 375487.619,
                "Y": 2777331.953,
                "Demand": 0,
                "W_S": 0,
                "W_F": 86400,
                "Service_time": 0,
                "N_V": nv,
                "C_V1": 1800,
                "C_V2": 1800,
                "C_V3": 1800,
            }

            df = pd.concat([pd.DataFrame([depot]), df], ignore_index=True)

            # 儲存檔案（命名為 001.csv, 002.csv...）
            file_number = str(i + 1).zfill(3)
            output_path = os.path.join(output_folder, f"{file_number}.csv")
            df.to_csv(output_path, index=False, encoding="utf-8-sig")
            print(f"✅ 已轉換: {filename} → {file_number}.csv（含瑞芳 depot）")


In [2]:
convert_files_with_depot("coordinations", "district_data")

✅ 已轉換: 台北市中山區.csv → 001.csv（含瑞芳 depot）
✅ 已轉換: 台北市中正區.csv → 002.csv（含瑞芳 depot）
✅ 已轉換: 台北市信義區.csv → 003.csv（含瑞芳 depot）
✅ 已轉換: 台北市內湖區.csv → 004.csv（含瑞芳 depot）
✅ 已轉換: 台北市北投區.csv → 005.csv（含瑞芳 depot）
✅ 已轉換: 台北市南京東.csv → 006.csv（含瑞芳 depot）
✅ 已轉換: 台北市南港區.csv → 007.csv（含瑞芳 depot）
✅ 已轉換: 台北市士林區.csv → 008.csv（含瑞芳 depot）
✅ 已轉換: 台北市大同區.csv → 009.csv（含瑞芳 depot）
✅ 已轉換: 台北市大安區.csv → 010.csv（含瑞芳 depot）
✅ 已轉換: 台北市文山區.csv → 011.csv（含瑞芳 depot）
✅ 已轉換: 台北市松山區.csv → 012.csv（含瑞芳 depot）
✅ 已轉換: 台北市萬華區.csv → 013.csv（含瑞芳 depot）
✅ 已轉換: 基隆市七堵區.csv → 014.csv（含瑞芳 depot）
✅ 已轉換: 基隆市中正區.csv → 015.csv（含瑞芳 depot）
✅ 已轉換: 基隆市仁愛區.csv → 016.csv（含瑞芳 depot）
✅ 已轉換: 基隆市信義區.csv → 017.csv（含瑞芳 depot）
✅ 已轉換: 基隆市安樂區.csv → 018.csv（含瑞芳 depot）
✅ 已轉換: 基隆市暖暖區.csv → 019.csv（含瑞芳 depot）
✅ 已轉換: 新北市三峽區.csv → 020.csv（含瑞芳 depot）
✅ 已轉換: 新北市三芝區.csv → 021.csv（含瑞芳 depot）
✅ 已轉換: 新北市三重區.csv → 022.csv（含瑞芳 depot）
✅ 已轉換: 新北市中和區.csv → 023.csv（含瑞芳 depot）
✅ 已轉換: 新北市五股區.csv → 024.csv（含瑞芳 depot）
✅ 已轉換: 新北市八里區.csv → 025.csv（含瑞芳 depot）
✅ 已轉換: 新北市土城區.csv → 026.c

##### Daily data

In [8]:
import pandas as pd
import random
from pathlib import Path
from collections import defaultdict

# 紀錄找不到的店名與對應日期
not_found_by_store = defaultdict(set)

# 步驟1：讀取 coordinates 對照表
coord_df = pd.read_csv('family_output_with_coordinates_XY.csv')
coord_dict = coord_df.groupby("name")[["X", "Y"]].first().to_dict(orient="index")

# 步驟2：讀取每日配送檔
folder_path = Path('Route Results 4')
csv_files = folder_path.glob("*.csv")

all_records = []

# 模糊比對 function
def fuzzy_match_coords(name, coord_dict):
    # 精確找
    if name in coord_dict:
        return coord_dict[name]
    if f"全家{name}" in coord_dict:
        return coord_dict[f"全家{name}"]
    
    # 模糊找（任一方向包含即可）
    for key in coord_dict.keys():
        if name in key or key in name:
            return coord_dict[key]
    return None


for file in csv_files:
    df = pd.read_csv(file)
    date_raw = str(df.iloc[0]["Date"]).zfill(4)  # e.g. '0307'
    date_fmt = f"2024{date_raw}"                # e.g. '20240307'
    N_V = df['Route Index'].max()

    for _, row in df.iterrows():
        name = row["Store Order"]
        coords = fuzzy_match_coords(name, coord_dict)
        if coords is None:
            not_found_by_store[name].add(date_fmt)
            continue
        X, Y = coords["X"], coords["Y"]
        is_dc = 1 if "ＤＣ" in name else 0
        demand = 0 if is_dc else random.randint(1, 15)
        all_records.append([
            date_fmt, name, X, Y, demand,
            0, 86400, 600, N_V,
            180, 180, 180
        ])

# 匯出結果
columns = ["Date", "Name", "X", "Y", "Demand", "W_S", "W_F", "Service_time", "N_V", "C_V1", "C_V2", "C_V3"]
output_df = pd.DataFrame(all_records, columns=columns)
output_df.to_csv("./Daily Data/formatted_all_routes_04.csv", index=False)

print("\n📌 找不到座標的店名與出現日期：")
for store, dates in not_found_by_store.items():
    print(f"{store}：{sorted(dates)}")


📌 找不到座標的店名與出現日期：
龜山忠義店：['20240401']
龜山光峰店：['20240401']
龜山興龍店：['20240401']
龜山萬壽店：['20240401']
龜山大傳店：['20240401']
龜山大同店：['20240401']
龜山德明店：['20240401']
龜山銘園店：['20240401']
龜山銘美店：['20240401']
龜山自強店：['20240401']
龜山中興店：['20240401']
龜山幸福店：['20240401']
龜山美滿店：['20240401']
龜山頂興店：['20240401']
龜山山鶯店：['20240401']
龜山興壽店：['20240401']
龜山陸光店：['20240401']
龜山幸美店：['20240401']


##### ALL data with XY

In [5]:
import pandas as pd
import os
from collections import defaultdict
from pathlib import Path

# 讀取座標對照表
coord_df = pd.read_csv("family_output_with_coordinates_XY.csv")
coord_dict = coord_df.groupby("name")[["X", "Y"]].first().to_dict(orient="index")

# 建立未匹配店名紀錄
not_found_by_store = defaultdict(set)

# 要讀取與寫入的資料夾
route_folder = Path("raw data/route")
output_folder = Path("route_with_xy")
output_folder.mkdir(exist_ok=True)

csv_files = [f for f in os.listdir(route_folder) if f.endswith(".csv")]

# 模糊比對函數
def fuzzy_match_coords(name, coord_dict):
    if name in coord_dict:
        return coord_dict[name]
    if f"全家{name}" in coord_dict:
        return coord_dict[f"全家{name}"]
    for key in coord_dict:
        if name in key or key in name:
            return coord_dict[key]
    return None

# 處理每個檔案
for filename in csv_files:
    file_path = route_folder / filename
    df = pd.read_csv(file_path)
    date = filename.split(".csv")[0]

    x_list, y_list = [], []
    for name in df["Store_Name"]:
        match = fuzzy_match_coords(name, coord_dict)
        if match:
            x_list.append(match["X"])
            y_list.append(match["Y"])
        else:
            x_list.append(None)
            y_list.append(None)
            not_found_by_store[name].add(date)

    df["X"] = x_list
    df["Y"] = y_list

    # 輸出含座標的檔案
    df.to_csv(output_folder / filename, index=False, encoding="utf-8-sig")


"📌 找不到座標的店名與出現日期：基隆中山店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2966', 'route_Car_KPD3801', 'route_Car_KPD3802', 'route_Car_KPD3805', 'route_Car_KPD3806', 'route_Car_KPD3923', 'route_Car_KPD3925', 'route_Car_KPD3927', 'route_Car_KPD3928', 'route_Car_KPD3951', 'route_TaskType_02', 'route_TaskType_03', 'route_TaskType_04', 'route_TaskType_05', 'route_TaskType_12']\n基隆華復店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2966', 'route_Car_KPD3801', 'route_Car_KPD3802', 'route_Car_KPD3805', 'route_Car_KPD3806', 'route_Car_KPD3923', 'route_Car_KPD3925', 'route_Car_KPD3927', 'route_Car_KPD3928', 'route_Car_KPD9998', 'route_TaskType_02', 'route_TaskType_03', 'route_TaskType_04', 'route_TaskType_05', 'route_TaskType_12']\n基隆經國店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2

In [6]:
# 印出找不到座標的店名與日期
# 按照車牌分類
import os
import pandas as pd

# 設定資料夾
folder_path = "route"

# 取得 CSV 檔案列表（使用 os）
csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

# 合併資料
all_data = []
for file in csv_files:
    file_path = os.path.join(folder_path, file)
    try:
        df = pd.read_csv(file_path)
        df["source_file"] = file
        all_data.append(df)
    except Exception as e:
        print(f"❌ 無法讀取 {file}: {e}")

# 合併為單一 DataFrame
df_all = pd.concat(all_data, ignore_index=True)

output_text = "\n📌 找不到座標的店名與出現日期："
for store, dates in not_found_by_store.items():
    output_text += f"{store}：{sorted(dates)}\n"

output_text.strip()

"📌 找不到座標的店名與出現日期：基隆中山店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2966', 'route_Car_KPD3801', 'route_Car_KPD3802', 'route_Car_KPD3805', 'route_Car_KPD3806', 'route_Car_KPD3923', 'route_Car_KPD3925', 'route_Car_KPD3927', 'route_Car_KPD3928', 'route_Car_KPD3951', 'route_TaskType_02', 'route_TaskType_03', 'route_TaskType_04', 'route_TaskType_05', 'route_TaskType_12']\n基隆華復店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2966', 'route_Car_KPD3801', 'route_Car_KPD3802', 'route_Car_KPD3805', 'route_Car_KPD3806', 'route_Car_KPD3923', 'route_Car_KPD3925', 'route_Car_KPD3927', 'route_Car_KPD3928', 'route_Car_KPD9998', 'route_TaskType_02', 'route_TaskType_03', 'route_TaskType_04', 'route_TaskType_05', 'route_TaskType_12']\n基隆經國店：['route_Car_KPD0226', 'route_Car_KPD0515', 'route_Car_KPD0578', 'route_Car_KPD0619', 'route_Car_KPD0626', 'route_Car_KPD2